In [ ]:
!pip install --upgrade transformers
!pip install --upgrade optimum
!pip install --upgrade datasets


!pip install --upgrade torch

!pip install --upgrade torchvision
!pip install --upgrade torchaudio

Resolved 40 packages in 2.02s
Prepared 1 package in 5.74s
Installed 1 package in 143ms
 + pillow==12.2.0


# Dataset Processing
1. Map the dhanmondi data with physician table for more information about the doctors.
these columns are taken for next process
```python
['PRSID', 'PHYID', 'PHYNM', 'PHYDEGR', 'PHY_SPC', 'PHY_DES', 'INS_NM', 'INS_ADD', 'INS_THA', 'INS_DST', 'CH_ADD', 'CH_DST', 'CH_THA', 'PHY_GEND', 'BMDC_REGNO', 'PHYNM_DT', 'CHNM_DT', 'IMAGE_PATH']
```
2. Pre-process images for fintuning (gray scale convertion, height,width ratio solve, contrast increase)

3. Prepare the image `ground truth` using mapped data.

## 1. Dataset mapping with physician table

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

import json
import random
import pandas as pd
import os
import re
import json
from os.path import join
from PIL import ImageEnhance
from PIL import Image

source_directory = '/content/drive/MyDrive/All_Image'
output_directory = '/content/drive/MyDrive/All_Image_Processed'
mapped_doctor_data= "/content/drive/MyDrive/Work_with_sazzad_vai/Dataset/mapped_doctor_data.csv"
jsonl_path = "/content/drive/MyDrive/Work_with_sazzad_vai/Dataset/ground_truth.jsonl"
data_dir = "/content/drive/MyDrive/Work_with_sazzad_vai/Dataset/"
max_width = 512
output_path = "data/processed"

### Processing the tables

In [ ]:


def extract_pr(path):
    if not isinstance(path, str):
        return None
    # Extract the filename part starting with PR (case insensitive just in case, but usually uppercase)
    match = re.search(r'(PR[A-Z0-9_]+)', os.path.basename(path), re.IGNORECASE)
    if match:
        return match.group(1)
    return None
def process_data(output_path):
    # Paths
    csv_path = "data/doctor_image_details (1).csv"
    excel_path = "data/R232C6_Dhanmondi_Data (2).xlsx"
    output_path = f"{output_path}/mapped_doctor_data.csv"

    print(f"Loading CSV: {csv_path}")
    df_csv = pd.read_csv(csv_path)
    print(f"CSV loaded. Sample data:\n{df_csv.head(2)}")

    # Extract PR from path
    # User says: "starting with pr is the pr"
    # Example: /content/drive/MyDrive/All_Image/PR232C6DHK23_P001.jpg -> PR232C6DHK23_P001
    

    df_csv['extracted_pr'] = df_csv['IMAGE_NAME_WITH_PATH'].apply(extract_pr)
    print(f"Extracted PR sample: {df_csv['extracted_pr'].head().tolist()}")

    print(f"Loading Excel: {excel_path}")
    # Load Excel - using the first sheet by default as checked before
    df_xl = pd.read_excel(excel_path)
    print(f"Excel loaded. Sample data:\n{df_xl.head(2)}")
    
    # Let's check for case insensitive match too
    df_xl['IMG_NM_upper'] = df_xl['IMG_NM'].astype(str).str.upper()
    df_csv['extracted_pr_upper'] = df_csv['extracted_pr'].astype(str).str.upper()

    print("Merging with Dhanmondi Data...")
    # Join on IMG_NM as it's the actual link between the files
    merged_df = pd.merge(
        df_csv, 
        df_xl, 
        left_on='extracted_pr_upper', 
        right_on='IMG_NM_upper', 
        how='left'
    )

    # Now load Physicianlist
    physician_list_path = "data/Physicianlist.xlsx"
    print(f"Loading Physician List: {physician_list_path}")
    df_physician = pd.read_excel(physician_list_path)
    
    # In Dhanmondi data it is 'PHYID', in Physicianlist it is 'PHY_ID'
    print("Merging with Physician List...")
    merged_df = pd.merge(
        merged_df,
        df_physician,
        left_on='PHYID',
        right_on='PHY_ID',
        how='left',
        suffixes=('', '_phy')
    )
    merged_df["IMAGE_PATH"] = merged_df['IMAGE_NAME_WITH_PATH'].str.split('/').str[-1]  # Keep only the filename for clarity
    # Clean up temporary columns and unwanted columns
    columns_to_drop = [
        'IMAGE_NAME_WITH_PATH','extracted_pr', 'extracted_pr_upper', 'IMG_NM_upper',
        'DOCTOR_DETAILS_COMBINED', 'MONTH', 'ROUND', 'YEAR', 'BOOKID', 'SHOPID', 
        'CDATE', 'PDATE', 'PRSTYPE', 'PSCSLNO', 'PHY_ID', 'PHY_NM', 'PHY_DEG',
        'VC2', 'NAME', 'GP', 'QTPRS', 'QTPURCH', 'CYCLE', 'FICODE', 'OPERATOR', 
        'DIAGCD', 'DIAGNAME', 'DIAGOPTR', 'DIAGEDTR', 'GENDER', 'AGE', 'PHYSPCD', 
        'CINSTCD', 'EDATE', 'ETIME', 'DIAEDATE', 'DIAETIME', 'SCHDSLT', 'FSCODE', 
        'EDITOR', 'EDDATE', 'ROUND_phy', 'CINSTCD_phy', 'MCODE', 'MARKET', 
        'PHYSP_C', 'FICODE_phy', 'SC', 'NOTE', 'PD03', 'PD04', 'DUPLICATE', 
        'OLDCODE', 'SHEETNO', 'EDITDATE', 'EDITOR_phy', 'UNICODE', 'CYCLE_phy', 
        'DSDCODE', 'MCHCODE', 'CH_PHNO1', 'CH_PHNO2', 'CH_PHNO3', 'PHY_PHNO', 
        'PHYEMAIL', 'PHYNM_DT_DUP', 'PHYNM_ALL_DUP', 'CHNM_DT_DUP', 'CHNM_ALL_DUP','IMG_NM','DIVISION','HINSTCD','OPERATO','REGION','DT'
    ]
    
    # Filter columns_to_drop to only those that exist in the dataframe
    existing_drops = [c for c in columns_to_drop if c in merged_df.columns]
    merged_df = merged_df.drop(columns=existing_drops)

    print(f"Merge complete. Rows in CSV: {len(df_csv)}, Rows in Merged: {len(merged_df)}")
    print(f"Matched rows in Dhanmondi: {merged_df['PRSID'].notna().sum() if 'PRSID' in merged_df.columns else 'N/A'}")
    print(f"Remaining columns: {merged_df.columns.tolist()}")

    # Save to CSV
    merged_df.to_csv(output_path, index=False)
    print(f"Result saved to: {output_path}")


In [21]:
output_path = "data/processed"
process_data(output_path)

## 2. Image Processing

In [ ]:


def process_image(image,max_width):
    """
    1. convert to gray scale
    2. resize to max_width while maintaining aspect ratio
    3. increase contrast
    """
    image = image.convert('L')  # Convert to grayscale
    
    if image.width > max_width:
        aspect_ratio = image.height / image.width
        new_height = int(max_width * aspect_ratio)
        image = image.resize((max_width, new_height))  # Resize while maintaining aspect ratio

    # Increase contrast
    image_enhanced = ImageEnhance.Contrast(image)
    image_enhanced = image_enhanced.enhance(2)  # Adjust the contrast level (2 is an example)
    return image_enhanced
    
def preprocess_images(source_dir, output_dir,max_width=512):
    image_paths = [os.path.join(source_dir, f) for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    image_paths = image_paths  # Process only the first 10 images
    for image_path in image_paths:
        image = Image.open(image_path)
        processed_image = process_image(image, max_width=max_width)
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, os.path.basename(image_path))
        processed_image.save(output_path,format='JPEG', quality=85, optimize=True)        

In [ ]:

preprocess_images(source_directory, output_directory)

## 3. Ground truth prepare

### Prepare table

In [34]:
df= pd.read_csv(mapped_doctor_data)
df.shape


(9640, 24)

In [35]:

print(df.shape)
df.drop_duplicates(inplace=True)
print(df.shape)

(9640, 24)
(1839, 24)


In [36]:
df.head()

,PRSID,PHYID,PHYNM,PHYDEGR,IMG_NM,DIVISION,PHY_SPC,PHY_DES,HINSTCD,INS_NM,...,CH_DST,CH_THA,OPERATO,REGION,DT,PHY_GEND,BMDC_REGNO,PHYNM_DT,CHNM_DT,image_path
0,PRS232C6016929,DHA28408,DR S M SIDDIQUR RAHMAN,"MBBS, D-CARD, MD, FACC",PR232C6DHK23_P001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PR232C6DHK23_P001.jpg
6,PRS232C6016937,DHK00929,DR MD. NUR HOSSAIN,"MBBS, MD",PR232C6DHK23_P002,1-DHK,CARDIOLOGY,PROFESSOR,H0320,NICVD,...,DHAKA,DHANMONDI,DILIP,DHK,DHAKADHANMONDI,M,NaN,DRMDNURHOSSAIN,MEDINOVA,PR232C6DHK23_P002.jpg
14,PRS232C6017997,DHK96865,DR MD FAIZUL ISLAM CHOWDHURY,"MBBS, FCPS, PHD, WHO",PR232C6DHK23_P003,1-DHK,MEDICINE,PROFESSOR,H0100,DHAKA MEDICAL COLLEGE HOSPITAL,...,DHAKA,DHANMONDI,4PDCD,DHK,DHAKADHANMONDI,M,A13255,DRMDFAIZULISLAMCHOWDHURY,MEDINOVA,PR232C6DHK23_P003.jpg
20,PRS232C6018008,DHK40007,DR MD. FAZLUL KADIR,"MBBS, FCPS",PR232C6DHK23_P004,1-DHK,MEDICINE,PROFESSOR,H0070,BANGLADESH MEDICAL COLLEGE HOSPITAL,...,DHAKA,DHANMONDI,4PBLS,DHK,DHAKADHANMONDI,M,NaN,DRMDFAZLULKADIR,MEDINOVA,PR232C6DHK23_P004.jpg
27,PRS232C6018026,DHK50102,DR M. ABU HENA CHOWDHURY,"MBBS, FCPS, DDV",PR232C6DHK23_P005,1-DHK,"SKIN, VD",ASSOCIATE PROFESSOR,H0110,"BSMMU, SHAHBAG",...,DHAKA,DHANMONDI,4PMRZ,DHK,DHAKADHANMONDI,M,NaN,DRMABUHENACHOWDHURY,IBN SINA MEDICAL IMAGING CENTER,PR232C6DHK23_P005.jpg


In [37]:
df.columns

Index(['PRSID', 'PHYID', 'PHYNM', 'PHYDEGR', 'IMG_NM', 'DIVISION', 'PHY_SPC',
       'PHY_DES', 'HINSTCD', 'INS_NM', 'INS_ADD', 'INS_THA', 'INS_DST',
       'CH_ADD', 'CH_DST', 'CH_THA', 'OPERATO', 'REGION', 'DT', 'PHY_GEND',
       'BMDC_REGNO', 'PHYNM_DT', 'CHNM_DT', 'image_path'],
      dtype='object')

In [38]:
# # check how many rows have  PRSID, PHYID, PHYNM and PHYDEGR only
## drop rows where all of these columns are null
df_pure = df.dropna(subset=['PHY_SPC', 'PHY_DES', 'INS_NM', 'INS_ADD', 'INS_THA'], how='all')
df_pure.shape
# df.dropna(df[df[['PHY_SPC', 'PHY_DES', 'INS_NM', 'INS_ADD', 'INS_THA']].isnull().all(axis=1)])

(1642, 24)

In [39]:
df_pure = df_pure.drop(columns=['PRSID',"CH_ADD", 'PHYID',"HINSTCD", 'PHY_DES', 'INS_NM',"PHYNM_DT","DT", 'INS_ADD', 'INS_THA', 'OPERATO',"IMG_NM",'BMDC_REGNO',"PHYNM_DT"], axis=1)
df_pure.shape

(1642, 11)

In [40]:
df_pure.head()

,PHYNM,PHYDEGR,DIVISION,PHY_SPC,INS_DST,CH_DST,CH_THA,REGION,PHY_GEND,CHNM_DT,image_path
6,DR MD. NUR HOSSAIN,"MBBS, MD",1-DHK,CARDIOLOGY,DHAKA,DHAKA,DHANMONDI,DHK,M,MEDINOVA,PR232C6DHK23_P002.jpg
14,DR MD FAIZUL ISLAM CHOWDHURY,"MBBS, FCPS, PHD, WHO",1-DHK,MEDICINE,DHAKA,DHAKA,DHANMONDI,DHK,M,MEDINOVA,PR232C6DHK23_P003.jpg
20,DR MD. FAZLUL KADIR,"MBBS, FCPS",1-DHK,MEDICINE,DHAKA,DHAKA,DHANMONDI,DHK,M,MEDINOVA,PR232C6DHK23_P004.jpg
27,DR M. ABU HENA CHOWDHURY,"MBBS, FCPS, DDV",1-DHK,"SKIN, VD",DHAKA,DHAKA,DHANMONDI,DHK,M,IBN SINA MEDICAL IMAGING CENTER,PR232C6DHK23_P005.jpg
34,DR MD NAZRUL ISLAM,"MBBS, FCPS, MD",1-DHK,"KIDNEY, MEDICINE",DHAKA,DHAKA,DHANMONDI,DHK,M,IBN SINA MEDICAL IMAGING CENTER,PR232C6DHK23_P007.jpg


In [41]:
df_pure.columns

Index(['PHYNM', 'PHYDEGR', 'DIVISION', 'PHY_SPC', 'INS_DST', 'CH_DST',
       'CH_THA', 'REGION', 'PHY_GEND', 'CHNM_DT', 'image_path'],
      dtype='object')

#### Finetune data format

In [42]:

# Output path for JSONL
exclude_fields = ["IMAGE_PATH","image_path", "PRSID", "PHYID"]
i = 0 
with open(jsonl_path, "w", encoding="utf-8") as f:
    for idx, row in df_pure.iterrows():

        # Convert row to dict and keep empty fields as empty string
        row_dict = {col: ("" if pd.isna(val) else val) for col, val in row.items()}
        # print(f"Processing row {idx}: {row_dict}")  # Debug print to check the content of each row
        row_dict
        i += 1
        record = {
            "idx": int(i),
            "image_path": row_dict.get("image_path", ""),
            "output": {k: v for k, v in row_dict.items() if k not in exclude_fields}
        }
        if i < 5:  # Print the first few records for verification
            print(f"Constructed record for row {i}: {record}")  # Debug print to check the constructed record

        f.write(json.dumps(record, ensure_ascii=False,default=str) + "\n")

print(f"JSONL file saved: {jsonl_path}")

Constructed record for row 1: {'idx': 1, 'image_path': 'PR232C6DHK23_P002.jpg', 'output': {'PHYNM': 'DR MD. NUR HOSSAIN', 'PHYDEGR': 'MBBS, MD', 'DIVISION': '1-DHK', 'PHY_SPC': 'CARDIOLOGY', 'INS_DST': 'DHAKA', 'CH_DST': 'DHAKA', 'CH_THA': 'DHANMONDI', 'REGION': 'DHK', 'PHY_GEND': 'M', 'CHNM_DT': 'MEDINOVA'}}
Constructed record for row 2: {'idx': 2, 'image_path': 'PR232C6DHK23_P003.jpg', 'output': {'PHYNM': 'DR MD FAIZUL ISLAM CHOWDHURY', 'PHYDEGR': 'MBBS, FCPS, PHD, WHO', 'DIVISION': '1-DHK', 'PHY_SPC': 'MEDICINE', 'INS_DST': 'DHAKA', 'CH_DST': 'DHAKA', 'CH_THA': 'DHANMONDI', 'REGION': 'DHK', 'PHY_GEND': 'M', 'CHNM_DT': 'MEDINOVA'}}
Constructed record for row 3: {'idx': 3, 'image_path': 'PR232C6DHK23_P004.jpg', 'output': {'PHYNM': 'DR MD. FAZLUL KADIR', 'PHYDEGR': 'MBBS, FCPS', 'DIVISION': '1-DHK', 'PHY_SPC': 'MEDICINE', 'INS_DST': 'DHAKA', 'CH_DST': 'DHAKA', 'CH_THA': 'DHANMONDI', 'REGION': 'DHK', 'PHY_GEND': 'M', 'CHNM_DT': 'MEDINOVA'}}
Constructed record for row 4: {'idx': 4, 'imag

### task : finetune data format


In [43]:
Task_prompt1 = """
You are a professional Medical Prescription Information Extractor.
Your rule to extract: doctor, institution, and chamber information from the prescription image.
Do not generate any introduction or conclusion.
""".strip()

Task_prompt2 = """
You are a professional Medical Prescription OCR Extractor.
Extract doctor, institution, and chamber information from the prescription image.
Extract the final output into a json format.
Do not generate any introduction or conclusion.
""".strip()



In [51]:
llm_finetune_data = []
train_ds= []
val_ds = []
image_path_set = set()

for line in open(jsonl_path, "r", encoding="utf-8"):
    if line.strip() == "":
        continue  # Skip empty lines
    # print(f"Reading line: {line.strip()}")  # Debug print to check the content of each line
    record = json.loads(line.strip())
    # print(f"Processing record idx: {record['idx']} with image_path: {record['image_path']}")  # Debug print to check the image path

    image_path = record["image_path"]
    image_path = os.path.join(output_directory, os.path.basename(image_path))  # Update to processed image path
    output = record["output"]

    if image_path in image_path_set:
        print(f"Duplicate image path found: {image_path}")
    else:

        image_path_set.add(image_path)

    task_1_sft_record = {
        "conversation": [
            {
                "value": "<image>"+Task_prompt1,
                "from": "human"
            },
            {
                "value": json.dumps(record["output"], ensure_ascii=False,default=str),
                "from": "gpt"
            }
                ],
        "images": [image_path]
        }
    task_2_sft_record = {
        "conversation": [
            {
                "value": "<image>"+Task_prompt2,
                "from": "human"
            },
            {         "value": json.dumps(record["output"], ensure_ascii=False,default=str),    "from": "gpt"            }
                ],  
        "images": [image_path]
    }
    llm_finetune_data.append(task_1_sft_record)
    llm_finetune_data.append(task_2_sft_record)
    random.Random(42).shuffle(llm_finetune_data)
    split_index = int(0.95 * len(llm_finetune_data))
    train_ds = llm_finetune_data[:split_index]
    val_ds = llm_finetune_data[split_index:]

In [52]:
len(train_ds), len(val_ds)

(3119, 165)

In [53]:
len(llm_finetune_data)

3284

### Save dataset

In [54]:

os.makedirs(
    join(data_dir, "llamafactory-ocr-finetune-data"), exist_ok=True
)

with open(join(data_dir, "llamafactory-ocr-finetune-data", "train-v1.json") , "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir, "llamafactory-ocr-finetune-data", "val-v1.json") , "w") as dest:
    json.dump(val_ds, dest, ensure_ascii=False, default=str)

In [55]:
join(data_dir, "llamafactory-ocr-finetune-data", "train-v1.json")



'/content/drive/MyDrive/Work_with_sazzad_vai/Dataset/llamafactory-ocr-finetune-data/train-v1.json'

## Finetune

In [56]:
%cd /content/drive/MyDrive/Work_with_sazzad_vai/training/LlamaFactory
!pwd

/content/drive/.shortcut-targets-by-id/1QUt0yYyYMFdS4PEmuVWZny7h11THdHJ6/Work_with_sazzad_vai/training/LlamaFactory
/content/drive/.shortcut-targets-by-id/1QUt0yYyYMFdS4PEmuVWZny7h11THdHJ6/Work_with_sazzad_vai/training/LlamaFactory


In [15]:

# !git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
# !cd LlamaFactory && git checkout 762b480131908d37736ad9aa3f12e87f8f7e6313

!pip install -e .
!pip install -r requirements/metrics.txt

Obtaining file:///content/drive/.shortcut-targets-by-id/1QUt0yYyYMFdS4PEmuVWZny7h11THdHJ6/Work_with_sazzad_vai/training/LlamaFactory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 38.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 121.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Train

In [ ]:
# from google.colab import userdata
# hf_token = userdata.get('huggingface')

!huggingface-cli login --token token


Hint: A new version of huggingface_hub (1.21.0) is available! You are using version 1.19.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [ ]:
# !pip install wandb weave
!wandb login token

wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [18]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00:00:0100:01


In [19]:
!pip install flash-linear-attention

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 703.0/703.0 kB 41.1 MB/s eta 0:00:00


In [ ]:
# !pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 66.1 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
  Running setup.py clean for flash-attn
Failed to build flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)


In [57]:
# %cd LlamaFactory
!export DISABLE_VERSION_CHECK=1 && llamafactory-cli train  examples/train_lora/ocr_finetune_qwen3_5_2b.yaml

[WARNING|2026-06-30 17:05:07] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-06-30 17:05:10] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:771] 2026-06-30 17:05:10,799 >> loading configuration file config.json from cache at /content/drive/MyDrive/Work_with_sazzad_vai/cache/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json
[INFO|configuration_utils.py:847] 2026-06-30 17:05:10,830 >> Model config Qwen3_5Config {
  "architectures": [
    "Qwen3_5ForConditionalGeneration"
  ],
  "image_token_id": 248056,
  "model_type": "qwen3_5",
  "text_config": {
    "attention_bias": false,
    "attention_dropout": 0.0,
    "attn_output_gate": true,
    "bos_token_id": null,
    "d

: 

In [1]:
!nvidia-smi

Thu Jun 18 18:07:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----